## Basic Chunking Stratergies

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")


def semantic_chunk(text, threshold):
    sentences = [s.strip() for s in text.split(".") if s.strip()]

    embeddings = model.encode(sentences)

    chunks = []
    current_chunk = [sentences[0]]

    for i in range(1, len(sentences)):
        sim = cosine_similarity(
            [embeddings[i - 1]],
            [embeddings[i]]
        )[0][0]

        if sim >= threshold:
            current_chunk.append(sentences[i])
        else:
            chunks.append(". ".join(current_chunk))
            current_chunk = [sentences[i]]

    chunks.append(". ".join(current_chunk))

    return chunks

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10692.08it/s]


In [23]:
text = """
Python is a programming language.
It is used for machine learning.
Pandas is a data analysis library.
The Eiffel Tower is in Paris.
France is a country in Europe.
"""

chunks = semantic_chunk(text,0.8)

for i, chunk in enumerate(chunks):
    print(f"\nChunk {i+1}")
    print(chunk)


Chunk 1
Python is a programming language

Chunk 2
It is used for machine learning

Chunk 3
Pandas is a data analysis library

Chunk 4
The Eiffel Tower is in Paris

Chunk 5
France is a country in Europe


## More Accuracy 

In [42]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")


def semantic_chunk_window(text, threshold):
    sentences = [s.strip() for s in text.split(".") if s.strip()]

    chunks = []
    current_chunk = [sentences[0]]

    for sentence in sentences[1:]:

        chunk_text = ". ".join(current_chunk)

        chunk_emb = model.encode(chunk_text)
        sent_emb = model.encode(sentence)

        sim = cosine_similarity(
            [chunk_emb],
            [sent_emb]
        )[0][0]

        if sim >= threshold:
            current_chunk.append(sentence)
        else:
            chunks.append(". ".join(current_chunk))
            current_chunk = [sentence]

    chunks.append(". ".join(current_chunk))

    return chunks

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11399.07it/s]


In [52]:
text = """
Python is a programming language.
It is used for machine learning.
Pandas is a data analysis library.
The Eiffel Tower is in Paris.
France is a country in Europe.
"""

chunks = semantic_chunk_window(text,0.2)

for i, chunk in enumerate(chunks):
    print(f"\nChunk {i+1}")
    print(chunk)


Chunk 1
Python is a programming language. It is used for machine learning. Pandas is a data analysis library

Chunk 2
The Eiffel Tower is in Paris. France is a country in Europe


In [49]:
text = """
    Apple released a new iPhone.
    It has 10 megaxiel ultrawide camera
"""

chunks = semantic_chunk_window(text,0.1)

for i, chunk in enumerate(chunks):
    print(f"\nChunk {i+1}")
    print(chunk)


Chunk 1
Apple released a new iPhone. It has 10 megaxiel ultrawide camera


## Using Langcian

In [ ]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={
        "device": "cpu"
    }
)

text_splitter = SemanticChunker(
    embeddings=embeddings
    
)

docs = text_splitter.create_documents([text])

for doc in docs:
    print(doc.page_content)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12028.10it/s]



Python is a programming language. It is used for machine learning. Pandas is a data analysis library. The Eiffel Tower is in Paris.
France is a country in Europe. 


## 4. Cross-Encoder Semantic Segmentation

In [ ]:
from sentence_transformers import CrossEncoder

model = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2", device="cpu"
)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 10292.65it/s]


[-9.635934]


In [27]:
score = model.predict([
    ("Apple released a new iPhone",
     "The phone contains an A18 chip")
])

print(score)

[-9.635934]


In [31]:
from sentence_transformers import CrossEncoder

model = CrossEncoder(
    "cross-encoder/stsb-roberta-base",
    device="cpu"
)

score = model.predict([
    ("Apple released a new iPhone",
     "The phone contains an A18 chip")
])

print(score)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 9110.27it/s]


[0.33685103]


In [ ]:
score = model.predict([
    ("Apple released a new iPhone",
     "It has 10 megaxiel ultrawide camera")
])

print(score)

[0.22233997]


In [29]:
from sentence_transformers import CrossEncoder

model = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    device="cpu"
)

score = model.predict([
    ("Apple released a new iPhone",
     "The phone contains an A18 chip")
])

print(score)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 11185.95it/s]


[-9.635934]


# Conclusion

- Semantic chunking output is not perfect 
- Cross Encoders maybe better but its output was also not very good, and its also expensive